# Raw Data Audit

## Objective

Audit every raw used-car CSV before cleaning or integration. The audit is non-destructive: it records schemas, missingness, duplicates, categories, ranges, hashes, and legacy-file overlap without changing the source files.

## Questions being answered

- Which files exist, and what are their exact schemas and inferred data types?
- Which source files are compatible with the canonical dataset?
- Are there missing values, whitespace issues, duplicates, invalid ranges, or unusual values?
- How much do `cclass.csv` and `focus.csv` overlap their manufacturer-level files?
- Which files should be included in the canonical source set?

## Imports

In [ ]:
import pandas as pd

from src.audit import run_audit
from src.config import REFERENCE_YEAR, TABLES_DIR

## Data-loading section

The reusable audit function reads only the explicit mappings in `src.config`. It does not use a blind CSV glob, and it writes audit outputs under `reports/tables/` and `reports/data_quality_report.md`.

In [ ]:
audit_outputs = run_audit()
print(f"Audit completed with reference year {REFERENCE_YEAR}.")
print(f"Tables written to: {TABLES_DIR}")

## Methodology

The audit creates a SHA-256 manifest, compares exact raw schemas, measures missing values and duplicates, trims strings only in temporary comparison copies, calculates numeric ranges and broad review flags, and compares the legacy files against their relevant manufacturer-file scopes. Review flags are diagnostic; they do not reject rows.

## Results: raw-data manifest

In [ ]:
audit_outputs["manifest"][["filename", "row_count", "column_count", "assigned_brand", "source_type", "included_in_canonical"]]

## Results: schema comparison

In [ ]:
audit_outputs["schema"][["filename", "tax_column", "mpg_available", "column_order_matches_complete", "schema_issues"]]

## Results: data quality

The candidate combined row count is reported separately from the individual source rows. No rows are removed by this audit.

In [ ]:
audit_outputs["quality"][["source", "row_count", "missing_cell_count", "exact_duplicate_rows", "zero_tax_count", "zero_engine_size_count", "future_year_count", "review_flag_count"]]

## Results: legacy-file overlap

In [ ]:
audit_outputs["legacy"][["legacy_file", "manufacturer_file", "legacy_row_count", "manufacturer_scope_row_count", "exact_matching_instances", "legacy_unmatched_instances", "recommendation"]]

## Interpretation

The nine manufacturer-level files are structurally compatible after explicit normalization of `tax(£)` in `hyundi.csv`. The two model-only files omit `tax` and `mpg` and overlap their manufacturer-level sources, so including them would risk artificial missingness and overrepresentation. Leading model whitespace is a loading concern, not a reason to edit raw files.

The year-2060 Ford record, zero engine-size values, unusual MPG values, and duplicate rows require documented decisions during the cleaning phase.

## Data-quality and analytical limitations

This is an audit of listing data, not confirmed sale transactions. There is no unique listing identifier, and the source context does not independently verify currency or measurement units. A quality flag does not prove that a record is invalid.

## Findings and Decisions

- Keep all raw files immutable.
- Use the nine explicit manufacturer-level sources for the canonical candidate.
- Keep `cclass.csv` and `focus.csv` outside the canonical dataset.
- Carry the audit outputs and row-level decisions into the cleaning phase.